# Tarea 4: Análisis de Datos y Optimización

## **Luis Angel Cupe Espinoza**

**Fecha de entrega:** 01 - 09 - 2026

**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa los tres problemas en este notebook
- Escribe tu código en las celdas indicadas
- Ejecuta todas las celdas para verificar que tu código funciona
- Guarda tu archivo `.ipynb` en la carpeta `tareas` de tu repositorio privado de GitHub (compartido con el docente)
- Envía el enlace a tu notebook en Canvas

**⚠️ IMPORTANTE:** GitHub registra el historial de cambios de cada archivo. Tu notebook debe ser subido a GitHub **antes del plazo**. **NO** modifiques el archivo después del plazo — los cambios tardíos serán detectados y pueden resultar en penalidad.

**Integridad académica:** Esta es una tarea individual. Puedes consultar los materiales del curso, documentación de Python, herramientas de IA y discutir conceptos con compañeros, pero todo el código debe ser tuyo.

---

In [54]:
# Importaciones estándar - ejecuta esta celda primero
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, curve_fit
from io import StringIO

---
## Problema 1: Análisis de Calidad de Agua en Ríos Andino-Amazónicos (7 puntos)

Estás analizando datos de calidad de agua de estaciones de monitoreo en tres ríos de la cuenca amazónica peruana, provenientes de la red de monitoreo de la **Autoridad Nacional del Agua (ANA)**. El conjunto de datos contiene mediciones de temperatura, oxígeno disuelto (OD), pH y conductividad eléctrica recolectadas durante varios meses.

### Tus Tareas:

**Parte A (2 puntos):** Carga y explora los datos
1. Carga los datos del string CSV provisto abajo en un DataFrame de pandas
2. Muestra información básica del conjunto de datos (forma, tipos de datos, primeras filas)
3. Verifica valores faltantes e indica cuántos hay en cada columna
4. Convierte la columna `fecha` a formato datetime usando `pd.to_datetime()`

**Parte B (3 puntos):** Análisis de datos con agrupación
1. Calcula la media, desviación estándar, mínimo y máximo de oxígeno disuelto (`od_mg_l`) agrupado por `estacion_id`
2. Determina qué estación tiene la media más baja de oxígeno disuelto
3. Crea una nueva columna llamada `estado_od` que clasifique cada medición como:
   - "Crítico" si OD < 4 mg/L
   - "Bajo" si OD está entre 4 y 6 mg/L
   - "Adecuado" si OD está entre 6 y 8 mg/L
   - "Bueno" si OD >= 8 mg/L
4. Cuenta cuántas mediciones caen en cada categoría de `estado_od` por estación

**Parte C (2 puntos):** Filtrado y resumen
1. Filtra los datos para incluir solo mediciones donde temperatura > 20°C Y pH entre 6.5 y 8.5
2. Para este subconjunto filtrado, calcula la conductividad media por mes (pista: extrae el mes de la fecha)
3. Identifica qué combinación estación-mes tuvo el mayor número de lecturas con OD "Crítico" o "Bajo"

In [55]:
# Dataset de calidad de agua - ríos andino-amazónicos del Perú
calidad_agua_csv = (
    "estacion_id,fecha,temp_c,od_mg_l,ph,conductividad_us\n"
    "RIO_UCAYALI,2024-05-15,24.3,7.2,7.1,145\n"
    "RIO_UCAYALI,2024-05-22,25.1,6.8,7.0,152\n"
    "RIO_UCAYALI,2024-06-05,24.8,6.5,6.9,158\n"
    "RIO_UCAYALI,2024-06-19,25.5,5.8,6.8,165\n"
    "RIO_UCAYALI,2024-07-03,24.2,5.2,7.0,172\n"
    "RIO_UCAYALI,2024-07-17,23.8,4.8,7.1,168\n"
    "RIO_UCAYALI,2024-08-01,24.5,4.2,7.2,175\n"
    "RIO_UCAYALI,2024-08-15,25.2,5.0,7.0,169\n"
    "RIO_TAMBOPATA,2024-05-15,21.8,8.5,7.4,98\n"
    "RIO_TAMBOPATA,2024-05-22,22.5,8.1,7.5,105\n"
    "RIO_TAMBOPATA,2024-06-05,22.9,7.8,7.3,112\n"
    "RIO_TAMBOPATA,2024-06-19,23.4,7.2,7.2,118\n"
    "RIO_TAMBOPATA,2024-07-03,22.1,6.8,7.1,125\n"
    "RIO_TAMBOPATA,2024-07-17,21.8,6.5,7.0,121\n"
    "RIO_TAMBOPATA,2024-08-01,22.5,6.9,7.1,128\n"
    "RIO_TAMBOPATA,2024-08-15,23.1,7.2,7.2,115\n"
    "RIO_MANTARO,2024-05-15,13.1,9.5,6.5,312\n"
    "RIO_MANTARO,2024-05-22,14.2,8.8,6.4,325\n"
    "RIO_MANTARO,2024-06-05,12.8,8.2,6.3,338\n"
    "RIO_MANTARO,2024-06-19,13.5,7.5,6.2,352\n"
    "RIO_MANTARO,2024-07-03,11.9,6.8,6.0,368\n"
    "RIO_MANTARO,2024-07-17,10.8,5.9,5.9,378\n"
    "RIO_MANTARO,2024-08-01,11.5,5.2,6.1,385\n"
    "RIO_MANTARO,2024-08-15,12.3,4.8,6.2,372\n"
)

# Parte A: Carga y explora los datos
df_agua = pd.read_csv(StringIO(calidad_agua_csv))

print("Forma del DataFrame (filas, columnas):", df_agua.shape)
print("\nTipos de datos por columna:")
print(df_agua.dtypes)
print("\nPrimeras filas:")
print(df_agua.head())

print("\nValores faltantes por columna:")
print(df_agua.isnull().sum())

# Conversión de la columna fecha a datetime
df_agua["fecha"] = pd.to_datetime(df_agua["fecha"])
print("\nTipo de 'fecha' tras la conversión:", df_agua["fecha"].dtype)


Forma del DataFrame (filas, columnas): (24, 6)

Tipos de datos por columna:
estacion_id             str
fecha                   str
temp_c              float64
od_mg_l             float64
ph                  float64
conductividad_us      int64
dtype: object

Primeras filas:
   estacion_id       fecha  temp_c  od_mg_l   ph  conductividad_us
0  RIO_UCAYALI  2024-05-15    24.3      7.2  7.1               145
1  RIO_UCAYALI  2024-05-22    25.1      6.8  7.0               152
2  RIO_UCAYALI  2024-06-05    24.8      6.5  6.9               158
3  RIO_UCAYALI  2024-06-19    25.5      5.8  6.8               165
4  RIO_UCAYALI  2024-07-03    24.2      5.2  7.0               172

Valores faltantes por columna:
estacion_id         0
fecha               0
temp_c              0
od_mg_l             0
ph                  0
conductividad_us    0
dtype: int64

Tipo de 'fecha' tras la conversión: datetime64[us]


In [56]:
# Parte B: Análisis de datos con agrupación

# 1. Estadísticas de OD agrupadas por estación
resumen_od = df_agua.groupby("estacion_id")["od_mg_l"].agg(
    media="mean", desviacion_std="std", minimo="min", maximo="max"
)
print("Resumen de oxígeno disuelto (OD) por estación:")
print(resumen_od)

# 2. Estación con la media más baja de OD
estacion_od_mas_bajo = resumen_od["media"].idxmin()
print(f"\nEstación con la media más baja de OD: {estacion_od_mas_bajo} "
      f"({resumen_od.loc[estacion_od_mas_bajo, 'media']:.2f} mg/L)")


# 3. Clasificación del estado de OD
def clasificar_od(od: float) -> str:
    """Clasifica una medición de oxígeno disuelto según su nivel.

    Args:
        od: Concentración de oxígeno disuelto en mg/L.

    Returns:
        Etiqueta de estado: "Crítico", "Bajo", "Adecuado" o "Bueno".
    """
    if od < 4:
        return "Crítico"
    elif od < 6:
        return "Bajo"
    elif od < 8:
        return "Adecuado"
    else:
        return "Bueno"


df_agua["estado_od"] = df_agua["od_mg_l"].apply(clasificar_od)
print("\nDataFrame con la columna 'estado_od':")
print(df_agua[["estacion_id", "fecha", "od_mg_l", "estado_od"]].head())

# 4. Conteo de mediciones por categoría de estado_od y estación
conteo_estado_od = (
    df_agua.groupby(["estacion_id", "estado_od"])
    .size()
    .unstack(fill_value=0)
)
print("\nConteo de mediciones por estado de OD y estación:")
print(conteo_estado_od)


Resumen de oxígeno disuelto (OD) por estación:
                media  desviacion_std  minimo  maximo
estacion_id                                          
RIO_MANTARO    7.0875        1.709166     4.8     9.5
RIO_TAMBOPATA  7.3750        0.692305     6.5     8.5
RIO_UCAYALI    5.6875        1.062931     4.2     7.2

Estación con la media más baja de OD: RIO_UCAYALI (5.69 mg/L)

DataFrame con la columna 'estado_od':
   estacion_id      fecha  od_mg_l estado_od
0  RIO_UCAYALI 2024-05-15      7.2  Adecuado
1  RIO_UCAYALI 2024-05-22      6.8  Adecuado
2  RIO_UCAYALI 2024-06-05      6.5  Adecuado
3  RIO_UCAYALI 2024-06-19      5.8      Bajo
4  RIO_UCAYALI 2024-07-03      5.2      Bajo

Conteo de mediciones por estado de OD y estación:
estado_od      Adecuado  Bajo  Bueno
estacion_id                         
RIO_MANTARO           2     3      3
RIO_TAMBOPATA         6     0      2
RIO_UCAYALI           3     5      0


In [57]:
# Parte C: Filtrado y resumen

# 1. Filtrar temperatura > 20°C y pH entre 6.5 y 8.5
filtro = (df_agua["temp_c"] > 20) & (df_agua["ph"].between(6.5, 8.5))
df_filtrado = df_agua[filtro].copy()
print(f"Registros que cumplen el filtro: {len(df_filtrado)} de {len(df_agua)}")

# 2. Conductividad media por mes en el subconjunto filtrado
df_filtrado["mes"] = df_filtrado["fecha"].dt.month
conductividad_por_mes = df_filtrado.groupby("mes")["conductividad_us"].mean()
print("\nConductividad media por mes (subconjunto filtrado):")
print(conductividad_por_mes)

# 3. Combinación estación-mes con más lecturas "Crítico" o "Bajo"
df_agua["mes"] = df_agua["fecha"].dt.month
mask_critico_bajo = df_agua["estado_od"].isin(["Crítico", "Bajo"])
conteo_estacion_mes = (
    df_agua[mask_critico_bajo]
    .groupby(["estacion_id", "mes"])
    .size()
    .sort_values(ascending=False)
)
print("\nConteo de lecturas 'Crítico'/'Bajo' por estación-mes:")
print(conteo_estacion_mes)

peor_combinacion = conteo_estacion_mes.index[0]
print(
    f"\nLa combinación con más lecturas críticas/bajas es "
    f"estación={peor_combinacion[0]}, mes={peor_combinacion[1]} "
    f"con {conteo_estacion_mes.iloc[0]} lecturas."
)



Registros que cumplen el filtro: 16 de 24

Conductividad media por mes (subconjunto filtrado):
mes
5    125.00
6    138.25
7    146.50
8    146.75
Name: conductividad_us, dtype: float64

Conteo de lecturas 'Crítico'/'Bajo' por estación-mes:
estacion_id  mes
RIO_MANTARO  8      2
RIO_UCAYALI  8      2
             7      2
RIO_MANTARO  7      1
RIO_UCAYALI  6      1
dtype: int64

La combinación con más lecturas críticas/bajas es estación=RIO_MANTARO, mes=8 con 2 lecturas.


---
## Problema 2: Comparación Estadística de Parcelas Forestales en la Amazonía (6 puntos)

Investigadores del **INIA (Instituto Nacional de Innovación Agraria) - Estación Experimental Pucallpa** midieron la biomasa arbórea (kg) en parcelas pareadas — unas sometidas a un tratamiento de aprovechamiento forestal de impacto reducido (AFIR) y otras dejadas como control. Se quiere determinar si el tratamiento afectó significativamente la biomasa individual de los árboles y si existe relación entre el diámetro y la biomasa.

### Tus Tareas:

**Parte A (2 puntos):** Comparación de grupos de tratamiento
1. Calcula estadísticas descriptivas (media, desviación estándar, mediana) de biomasa para cada grupo
2. Realiza una prueba t de dos muestras independientes para determinar si hay diferencia significativa en la biomasa media entre parcelas control y AFIR (α = 0.05)
3. Plantea tu hipótesis nula y alternativa, reporta el estadístico t y el p-valor, y escribe una conclusión

**Parte B (2 puntos):** Análisis de correlación
1. Calcula el coeficiente de correlación de Pearson entre el DAP y la biomasa para todo el conjunto de datos
2. Evalúa si esta correlación es estadísticamente significativa (α = 0.05)
3. Interpreta la fuerza y dirección de la correlación

**Parte C (2 puntos):** Ajuste de distribución
1. Ajusta una distribución normal a los datos de biomasa de las parcelas control
2. Reporta los parámetros ajustados (μ y σ)
3. Calcula la probabilidad de que un árbol seleccionado aleatoriamente de las parcelas control tenga biomasa > 150 kg
4. ¿Qué valor de biomasa representa el percentil 90 para los árboles de parcelas control?

In [58]:
# Datos de parcelas forestales - INIA Pucallpa
np.random.seed(458)  # Para reproducibilidad

# Parcelas control: bosque sin intervención
n_control = 35
dap_control = np.random.uniform(15, 50, n_control)  # DAP en cm
biomasa_control = 0.1 * dap_control**2.2 + np.random.normal(0, 15, n_control)
biomasa_control = np.maximum(biomasa_control, 10)  # Asegurar valores positivos

# Parcelas AFIR: los árboles remanentes disponen de más recursos
n_afir = 30
dap_afir = np.random.uniform(18, 55, n_afir)  # DAP en cm
biomasa_afir = 0.12 * dap_afir**2.2 + np.random.normal(5, 18, n_afir)
biomasa_afir = np.maximum(biomasa_afir, 10)

# Crear DataFrame
bosque_df = pd.DataFrame({
    'dap_cm': np.concatenate([dap_control, dap_afir]),
    'biomasa_kg': np.concatenate([biomasa_control, biomasa_afir]),
    'tratamiento': ['Control']*n_control + ['AFIR']*n_afir
})

print(bosque_df.head())
print(f"\nEspecies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)")

      dap_cm  biomasa_kg tratamiento
0  43.208835  395.956074     Control
1  49.475851  521.291644     Control
2  19.647405   54.799539     Control
3  23.651882  119.264060     Control
4  40.431164  335.576313     Control

Especies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)


In [59]:
# Parte A: Comparación de grupos de tratamiento

# 1. Estadísticas descriptivas de biomasa por grupo
resumen_biomasa = bosque_df.groupby("tratamiento")["biomasa_kg"].agg(
    media="mean", desviacion_std="std", mediana="median"
)
print("Estadísticas descriptivas de biomasa por tratamiento:")
print(resumen_biomasa)

# 2. Prueba t de dos muestras independientes
biomasa_control_arr = bosque_df.loc[
    bosque_df["tratamiento"] == "Control", "biomasa_kg"
]
biomasa_afir_arr = bosque_df.loc[
    bosque_df["tratamiento"] == "AFIR", "biomasa_kg"
]

t_stat, p_valor = stats.ttest_ind(
    biomasa_control_arr, biomasa_afir_arr, equal_var=False
)

print(f"\nEstadístico t: {t_stat:.4f}")
print(f"p-valor: {p_valor:.4f}")

# 3. Hipótesis y conclusión
alpha = 0.05
print(
    "\nH0: La biomasa media de los árboles no difiere entre "
    "parcelas Control y AFIR (μ_control = μ_afir)."
)
print(
    "H1: La biomasa media de los árboles difiere entre "
    "parcelas Control y AFIR (μ_control ≠ μ_afir)."
)

if p_valor < alpha:
    conclusion = (
        f"Como p-valor ({p_valor:.4f}) < α ({alpha}), se rechaza H0: "
        "existe evidencia estadística de una diferencia significativa "
        "en la biomasa media entre parcelas Control y AFIR."
    )
else:
    conclusion = (
        f"Como p-valor ({p_valor:.4f}) >= α ({alpha}), no se rechaza H0: "
        "no hay evidencia estadística suficiente de una diferencia "
        "significativa en la biomasa media entre ambos tratamientos."
    )
print("\nConclusión:", conclusion)




Estadísticas descriptivas de biomasa por tratamiento:
                  media  desviacion_std     mediana
tratamiento                                        
AFIR         325.418953      213.516434  239.121336
Control      269.642640      158.210017  232.870089

Estadístico t: -1.1799
p-valor: 0.2433

H0: La biomasa media de los árboles no difiere entre parcelas Control y AFIR (μ_control = μ_afir).
H1: La biomasa media de los árboles difiere entre parcelas Control y AFIR (μ_control ≠ μ_afir).

Conclusión: Como p-valor (0.2433) >= α (0.05), no se rechaza H0: no hay evidencia estadística suficiente de una diferencia significativa en la biomasa media entre ambos tratamientos.


In [60]:
# Parte B: Análisis de correlación

# 1. Coeficiente de correlación de Pearson entre DAP y biomasa
r_pearson, p_valor_corr = stats.pearsonr(
    bosque_df["dap_cm"], bosque_df["biomasa_kg"]
)
print(f"Coeficiente de correlación de Pearson (r): {r_pearson:.4f}")
print(f"p-valor asociado: {p_valor_corr:.6f}")

# 2. Significancia estadística (α = 0.05)
alpha = 0.05
if p_valor_corr < alpha:
    print(
        f"\nComo p-valor ({p_valor_corr:.6f}) < α ({alpha}), la correlación "
        "es estadísticamente significativa."
    )
else:
    print(
        f"\nComo p-valor ({p_valor_corr:.6f}) >= α ({alpha}), la correlación "
        "no es estadísticamente significativa."
    )

# 3. Interpretación de fuerza y dirección
if abs(r_pearson) >= 0.7:
    fuerza = "fuerte"
elif abs(r_pearson) >= 0.4:
    fuerza = "moderada"
else:
    fuerza = "débil"
direccion = "positiva" if r_pearson > 0 else "negativa"

print(
    f"\nInterpretación: existe una correlación {fuerza} y {direccion} "
    f"entre el DAP y la biomasa (r = {r_pearson:.4f}). Esto es consistente "
    "con la relación alométrica esperada: a mayor diámetro del árbol, "
    "mayor biomasa."
)


Coeficiente de correlación de Pearson (r): 0.9565
p-valor asociado: 0.000000

Como p-valor (0.000000) < α (0.05), la correlación es estadísticamente significativa.

Interpretación: existe una correlación fuerte y positiva entre el DAP y la biomasa (r = 0.9565). Esto es consistente con la relación alométrica esperada: a mayor diámetro del árbol, mayor biomasa.


In [61]:
# Parte C: Ajuste de distribución

# 1. Ajustar una distribución normal a la biomasa de parcelas control
mu_control, sigma_control = stats.norm.fit(biomasa_control_arr)

# 2. Parámetros ajustados
print(f"Parámetros ajustados de la distribución normal (parcelas Control):")
print(f"  μ (media)             = {mu_control:.4f} kg")
print(f"  σ (desviación típica) = {sigma_control:.4f} kg")

# 3. P(biomasa > 150 kg) usando la función de supervivencia (1 - CDF)
prob_mayor_150 = stats.norm.sf(150, loc=mu_control, scale=sigma_control)
print(
    f"\nP(biomasa > 150 kg) = {prob_mayor_150:.4f} "
    f"({prob_mayor_150 * 100:.2f}%)"
)

# 4. Percentil 90 (valor de biomasa que deja el 90% de los datos por debajo)
percentil_90 = stats.norm.ppf(0.90, loc=mu_control, scale=sigma_control)
print(f"Percentil 90 de biomasa (Control): {percentil_90:.2f} kg")




Parámetros ajustados de la distribución normal (parcelas Control):
  μ (media)             = 269.6426 kg
  σ (desviación típica) = 155.9335 kg

P(biomasa > 150 kg) = 0.7785 (77.85%)
Percentil 90 de biomasa (Control): 469.48 kg


---
## Problema 3: Ajuste de Curva de Respuesta a la Luz (7 puntos)

La fotosíntesis depende de la intensidad de luz siguiendo una curva de saturación. La **hipérbola rectangular** se usa comúnmente para modelar esta relación:

$$A = \frac{A_{max} \cdot I}{K + I} - R_d$$

Donde:
- $A$ = tasa de fotosíntesis neta (μmol CO₂ m⁻² s⁻¹)
- $A_{max}$ = tasa máxima de fotosíntesis a saturación de luz
- $I$ = intensidad de luz (μmol fotones m⁻² s⁻¹, PAR)
- $K$ = constante de media saturación (nivel de luz en el que A = A_max/2 - R_d)
- $R_d$ = tasa de respiración en oscuridad (CO₂ liberado cuando I = 0)

Los datos provienen de mediciones de *Cecropia sp.* ("cetico"), una especie pionera característica de la Amazonía peruana, muy importante en la regeneración de bosques perturbados.

### Tus Tareas:

**Parte A (2 puntos):** Define el modelo y la función de costo
1. Escribe una función `respuesta_luz(I, Amax, K, Rd)` que implemente la ecuación anterior
2. Escribe una función de costo `respuesta_luz_mse(params, I_datos, A_datos)` que calcule el error cuadrático medio entre las tasas de fotosíntesis observadas y predichas
3. Prueba tu función `respuesta_luz` calculando A para I = 500 con Amax=25, K=200, Rd=2

**Parte B (3 puntos):** Ajusta el modelo usando optimización
1. Usa `scipy.optimize.minimize` para encontrar los parámetros óptimos (Amax, K, Rd) que minimicen el MSE
2. Usa valores iniciales: Amax=20, K=150, Rd=1
3. Reporta los parámetros ajustados y el MSE final
4. Ajusta también el modelo usando `scipy.optimize.curve_fit` y compara los resultados

**Parte C (2 puntos):** Evalúa e interpreta el modelo
1. Calcula los valores de fotosíntesis predichos usando tus parámetros ajustados
2. Calcula R² (coeficiente de determinación) para evaluar el ajuste del modelo:
   $$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
3. Calcula el **punto de compensación lumínico** (el nivel de luz donde A = 0, es decir, la fotosíntesis iguala a la respiración). Pista: despeja I cuando A = 0
4. ¿Cuál es la tasa de fotosíntesis a saturación lumínica (Amax - Rd)?

In [62]:
# Datos de curva de respuesta a la luz - Cecropia sp. (cetico)
# Mediciones en parcela de investigación, Madre de Dios

# PAR (radiación fotosintéticamente activa) en μmol fotones m⁻² s⁻¹
par_datos = np.array([0, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000, 1200, 1500, 1800])

# Tasa de fotosíntesis neta en μmol CO₂ m⁻² s⁻¹
foto_datos = np.array([-1.8, 1.2, 4.5, 7.1, 9.2, 12.5, 14.8, 17.5, 19.2, 21.1, 22.0, 22.5, 22.8, 23.0, 23.1])

print(f"Rango PAR: {par_datos.min()} a {par_datos.max()} μmol fotones m⁻² s⁻¹")
print(f"Rango fotosíntesis: {foto_datos.min()} a {foto_datos.max()} μmol CO₂ m⁻² s⁻¹")
print("Especie: Cecropia sp. (cetico) - pionera amazónica")

Rango PAR: 0 a 1800 μmol fotones m⁻² s⁻¹
Rango fotosíntesis: -1.8 a 23.1 μmol CO₂ m⁻² s⁻¹
Especie: Cecropia sp. (cetico) - pionera amazónica


In [63]:
# Parte A: Define el modelo y la función de costo

def respuesta_luz(I, Amax, K, Rd):
    """Modelo de hipérbola rectangular para la respuesta fotosintética a la luz.

    Args:
        I: Intensidad de luz (PAR) en μmol fotones m⁻² s⁻¹. Puede ser un
            escalar o un array de numpy.
        Amax: Tasa máxima de fotosíntesis a saturación de luz.
        K: Constante de media saturación.
        Rd: Tasa de respiración en oscuridad.

    Returns:
        Tasa de fotosíntesis neta predicha (misma forma que ``I``).
    """
    return (Amax * I) / (K + I) - Rd


def respuesta_luz_mse(params, I_datos, A_datos):
    """Error cuadrático medio entre la fotosíntesis observada y predicha.

    Args:
        params: Secuencia (Amax, K, Rd) con los parámetros del modelo.
        I_datos: Array con las intensidades de luz observadas.
        A_datos: Array con las tasas de fotosíntesis observadas.

    Returns:
        Error cuadrático medio (float).
    """
    Amax, K, Rd = params
    A_predicho = respuesta_luz(I_datos, Amax, K, Rd)
    return np.mean((A_datos - A_predicho) ** 2)


# 3. Prueba de la función respuesta_luz
A_prueba = respuesta_luz(500, Amax=25, K=200, Rd=2)
print(f"respuesta_luz(I=500, Amax=25, K=200, Rd=2) = {A_prueba:.4f} "
      f"μmol CO₂ m⁻² s⁻¹")




respuesta_luz(I=500, Amax=25, K=200, Rd=2) = 15.8571 μmol CO₂ m⁻² s⁻¹


In [64]:
# Parte B: Ajusta el modelo usando optimización

# 1-3. Ajuste con scipy.optimize.minimize
params_iniciales = [20, 150, 1]  # Amax, K, Rd

resultado_min = minimize(
    respuesta_luz_mse,
    x0=params_iniciales,
    args=(par_datos, foto_datos),
    method="Nelder-Mead",
)

Amax_min, K_min, Rd_min = resultado_min.x
mse_min = resultado_min.fun

print("Resultado con scipy.optimize.minimize:")
print(f"  Amax = {Amax_min:.4f}")
print(f"  K    = {K_min:.4f}")
print(f"  Rd   = {Rd_min:.4f}")
print(f"  MSE final = {mse_min:.4f}")

# 4. Ajuste con scipy.optimize.curve_fit
params_curve_fit, _covarianza = curve_fit(
    respuesta_luz, par_datos, foto_datos, p0=params_iniciales
)
Amax_cf, K_cf, Rd_cf = params_curve_fit
mse_cf = respuesta_luz_mse(params_curve_fit, par_datos, foto_datos)

print("\nResultado con scipy.optimize.curve_fit:")
print(f"  Amax = {Amax_cf:.4f}")
print(f"  K    = {K_cf:.4f}")
print(f"  Rd   = {Rd_cf:.4f}")
print(f"  MSE final = {mse_cf:.4f}")

print(
    "\nComparación: ambos métodos convergen a parámetros muy similares, "
    "ya que minimizan la misma función de costo (MSE); curve_fit usa "
    "mínimos cuadrados no lineales (Levenberg-Marquardt), que suele ser "
    "más rápido y estable para este tipo de problema que Nelder-Mead."
)


Resultado con scipy.optimize.minimize:
  Amax = 28.4460
  K    = 134.7544
  Rd   = 2.6272
  MSE final = 0.2373

Resultado con scipy.optimize.curve_fit:
  Amax = 28.4460
  K    = 134.7545
  Rd   = 2.6272
  MSE final = 0.2373

Comparación: ambos métodos convergen a parámetros muy similares, ya que minimizan la misma función de costo (MSE); curve_fit usa mínimos cuadrados no lineales (Levenberg-Marquardt), que suele ser más rápido y estable para este tipo de problema que Nelder-Mead.


In [65]:
# Parte C: Evalúa e interpreta el modelo

# Se usan los parámetros de curve_fit por ser el método estándar para
# ajuste de curvas no lineales.
Amax_final, K_final, Rd_final = Amax_cf, K_cf, Rd_cf

# 1. Valores predichos
foto_predicho = respuesta_luz(par_datos, Amax_final, K_final, Rd_final)

# 2. Coeficiente de determinación R²
ss_res = np.sum((foto_datos - foto_predicho) ** 2)
ss_tot = np.sum((foto_datos - np.mean(foto_datos)) ** 2)
r_cuadrado = 1 - (ss_res / ss_tot)
print(f"R² del modelo ajustado: {r_cuadrado:.4f}")

# 3. Punto de compensación lumínico: A = 0 => Amax*I/(K+I) - Rd = 0
# Despejando I: I = Rd * K / (Amax - Rd)
punto_compensacion = (Rd_final * K_final) / (Amax_final - Rd_final)
print(f"Punto de compensación lumínico: {punto_compensacion:.2f} "
      "μmol fotones m⁻² s⁻¹")

# 4. Fotosíntesis neta a saturación lumínica
foto_saturacion = Amax_final - Rd_final
print(f"Fotosíntesis neta a saturación lumínica (Amax - Rd): "
      f"{foto_saturacion:.4f} μmol CO₂ m⁻² s⁻¹")

print(
    "\nInterpretación: el modelo explica "
    f"{r_cuadrado * 100:.2f}% de la variabilidad observada en la tasa de "
    "fotosíntesis neta. Por debajo del punto de compensación lumínico, la "
    "planta respira más CO₂ del que fija por fotosíntesis (A < 0); por "
    "encima, es fotosintéticamente productiva."
)


R² del modelo ajustado: 0.9966
Punto de compensación lumínico: 13.71 μmol fotones m⁻² s⁻¹
Fotosíntesis neta a saturación lumínica (Amax - Rd): 25.8189 μmol CO₂ m⁻² s⁻¹

Interpretación: el modelo explica 99.66% de la variabilidad observada en la tasa de fotosíntesis neta. Por debajo del punto de compensación lumínico, la planta respira más CO₂ del que fija por fotosíntesis (A < 0); por encima, es fotosintéticamente productiva.


---
## Lista de verificación para entrega

Antes de entregar, verifica que:

- [ ] Todas las celdas de código se ejecutan sin errores
- [ ] Los tres problemas están completos
- [ ] Los resultados son visibles en todas las celdas
- [ ] Tu nombre está incluido al inicio
- [ ] El archivo está guardado en la carpeta `tareas` de tu repositorio privado de GitHub
- [ ] El archivo está subido a GitHub **antes del plazo**
- [ ] El enlace a tu notebook está enviado en Canvas